# Composite Panel MDO

Working notes on `composite_panel` and `optimize_laminate.py` — covers the full stack from material constants through CLT, failure criteria, aero loads, and minimum-mass optimisation.

**Sections:**
1. Material properties — what defines a composite ply
2. The Q-bar matrix — rotating ply stiffness into a global frame
3. Classical Laminate Theory — building the ABD matrix
4. CLT response — strains, curvatures, and ply stresses from applied loads
5. Tsai-Wu failure criterion — when does a ply fail?
6. Aerodynamic loads — from flight condition to running loads
7. The optimizer — minimum-mass laminate design
8. Wing-level sizing — panel-by-panel across the semi-span
9. Pareto sweep — trading mass against structural margin

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # find composite_panel package

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Dark plot style consistent with the demo figures
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#0a0e1a',
    'axes.edgecolor':   '#3a4060',
    'axes.labelcolor':  '#e8edf5',
    'xtick.color':      '#e8edf5',
    'ytick.color':      '#e8edf5',
    'text.color':       '#e8edf5',
    'grid.color':       '#3a4060',
    'grid.alpha':       0.5,
})

BLUE  = '#00aaff'; GOLD = '#f0a030'; RED = '#ff4455'
TEAL  = '#00ddbb'; WHITE = '#e8edf5'; DIM = '#3a4060'

print('Setup complete.')

---
## 1. Material Properties

A unidirectional carbon-epoxy ply is defined by **five elastic constants** and **five strength values**.

### Elastic constants
| Symbol | Meaning | IM7/8552 value |
|--------|---------|----------------|
| E1 | Young's modulus along fibres | 171.4 GPa |
| E2 | Young's modulus ⊥ fibres | 9.08 GPa |
| G12 | In-plane shear modulus | 5.29 GPa |
| ν12 | Major Poisson's ratio | 0.32 |
| ν21 = ν12·E2/E1 | Minor Poisson's ratio | ~0.017 |

### Strength allowables
| Symbol | Meaning | IM7/8552 value |
|--------|---------|----------------|
| F1t | Longitudinal tensile strength | 2326 MPa |
| F1c | Longitudinal compressive strength | 1200 MPa |
| F2t | Transverse tensile strength | 62 MPa |
| F2c | Transverse compressive strength | 200 MPa |
| F12 | In-plane shear strength | 110 MPa |

Notice how **weak** F2t is (62 MPa) compared to F1t (2326 MPa). The matrix is the weak link.
This is why we always orient at least some plies off-axis — to share transverse loads between fibre directions.

Reference: Hexcel HexPly 8552 product data sheet (2016)

In [ ]:
from composite_panel import IM7_8552

mat = IM7_8552()

print('IM7/8552 elastic constants:')
print(f'  E1  = {mat.E1/1e9:.1f} GPa   (fibre direction)')
print(f'  E2  = {mat.E2/1e9:.2f} GPa  (transverse)')
print(f'  G12 = {mat.G12/1e9:.2f} GPa  (in-plane shear)')
print(f'  nu12 = {mat.nu12:.2f}')
print(f'  nu21 = {mat.nu12 * mat.E2 / mat.E1:.4f}  (derived: nu21 = nu12*E2/E1)')
print()
print('Strength allowables:')
print(f'  F1t = {mat.F1t/1e6:.0f} MPa  F1c = {mat.F1c/1e6:.0f} MPa')
print(f'  F2t = {mat.F2t/1e6:.0f} MPa   F2c = {mat.F2c/1e6:.0f} MPa')
print(f'  F12 = {mat.F12/1e6:.0f} MPa')
print()
print(f'Fibre/transverse tensile strength ratio: {mat.F1t/mat.F2t:.0f}x')
print(f'  → A 0° ply can carry {mat.F1t/mat.F2t:.0f}x more load along the fibres than across them')

In [ ]:
# Visualise the anisotropy ratio — how much stiffer is E1 vs E2?
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Stiffness bar chart
ax = axes[0]
names = ['E1\n(fibre)', 'E2\n(transverse)', 'G12\n(shear)']
values = [mat.E1/1e9, mat.E2/1e9, mat.G12/1e9]
colors = [BLUE, RED, GOLD]
bars = ax.bar(names, values, color=colors, edgecolor=DIM, width=0.5)
ax.bar_label(bars, fmt='%.1f GPa', color=WHITE, fontsize=9, padding=3)
ax.set_title('Elastic moduli — IM7/8552', fontsize=11)
ax.set_ylabel('Modulus [GPa]')
ax.set_ylim(0, 200)

# Strength bar chart
ax = axes[1]
s_names = ['F1t', 'F1c', 'F2t', 'F2c', 'F12']
s_vals  = [mat.F1t/1e6, mat.F1c/1e6, mat.F2t/1e6, mat.F2c/1e6, mat.F12/1e6]
s_cols  = [BLUE, RED, TEAL, '#ff8833', GOLD]
bars = ax.bar(s_names, s_vals, color=s_cols, edgecolor=DIM, width=0.5)
ax.bar_label(bars, fmt='%.0f MPa', color=WHITE, fontsize=9, padding=3)
ax.set_title('Strength allowables — IM7/8552', fontsize=11)
ax.set_ylabel('Strength [MPa]')
ax.set_ylim(0, 2700)
ax.annotate('Weak transverse\ndirection!', xy=(2, mat.F2t/1e6), xytext=(2.5, 600),
            arrowprops=dict(arrowstyle='->', color=RED), color=RED, fontsize=9)

plt.tight_layout()
plt.show()

---
## 2. The Q-Bar Matrix — Rotating Ply Stiffness

A single ply oriented at angle **θ** to the laminate x-axis contributes to the laminate stiffness
through its **rotated reduced stiffness matrix Q̄** (pronounced "Q-bar").

### Why Q̄ and not just Q?

The reduced stiffness matrix **Q** describes a ply's behaviour in its own (1-2) frame:
```
{σ₁₂} = [Q] {ε₁₂}       (material frame)
```
But in a laminate, all plies share the same global strain (x-y frame). We need Q in the global frame:
```
{σ_xy} = [Q̄(θ)] {ε_xy}  (global frame)
```

### The invariant form (what the code uses)

Rather than computing T⁻¹QTᵀ⁻¹ (which needs a matrix inverse), the code uses the
**Tsai-Pagano invariant form** — Q̄ expressed directly as polynomials in cos(θ) and sin(θ):

$$\bar{Q}_{11} = Q_{11}c^4 + 2(Q_{12}+2Q_{66})c^2s^2 + Q_{22}s^4$$
$$\bar{Q}_{22} = Q_{11}s^4 + 2(Q_{12}+2Q_{66})c^2s^2 + Q_{22}c^4$$

This is just arithmetic — no matrix inversions — and is differentiable with respect to θ.
That differentiability is essential when θ is an optimization variable.

Reference: Kassapoglou (2013) §2.4

In [ ]:
# Compute Q̄ for a range of angles and plot key components
# This shows how ply stiffness rotates as you change orientation

# Import the internal helper (underscore = private, but we're using it for education)
import importlib.util, types
# optimizer is now part of composite_panel package
opt_mod = importlib.util.load_from_spec = spec

# Direct import
import sys
sys.path.insert(0, '.')
from composite_panel.optimizer import _Q_bar_matrix

angles_deg = np.linspace(0, 90, 181)
Qb11, Qb22, Qb12, Qb66, Qb16 = [], [], [], [], []

for theta_deg in angles_deg:
    Qb = _Q_bar_matrix(mat, np.radians(theta_deg))
    Qb11.append(float(Qb[0,0]) / 1e9)
    Qb22.append(float(Qb[1,1]) / 1e9)
    Qb12.append(float(Qb[0,1]) / 1e9)
    Qb66.append(float(Qb[2,2]) / 1e9)
    Qb16.append(float(Qb[0,2]) / 1e9)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(angles_deg, Qb11, color=BLUE,  lw=2, label='Q̄₁₁  (axial stiffness in x)')
ax.plot(angles_deg, Qb22, color=RED,   lw=2, label='Q̄₂₂  (axial stiffness in y)')
ax.plot(angles_deg, Qb66, color=GOLD,  lw=2, label='Q̄₆₆  (shear stiffness)')
ax.plot(angles_deg, Qb16, color=TEAL,  lw=1.5, linestyle='--', label='Q̄₁₆  (extension-shear coupling)')

ax.axvline(45, color=WHITE, lw=0.8, linestyle=':', alpha=0.5)
ax.annotate('±45° maximises\nshear stiffness Q̄₆₆', xy=(45, max(Qb66)),
            xytext=(55, max(Qb66)*1.05), color=GOLD, fontsize=8,
            arrowprops=dict(arrowstyle='->', color=GOLD))
ax.annotate('Q̄₁₆ = 0 at 0° and 90°\n(no extension-shear coupling)', xy=(0, 0),
            xytext=(20, -15), color=TEAL, fontsize=8)

ax.set_xlabel('Ply angle θ [deg]')
ax.set_ylabel('Q̄ component [GPa]')
ax.set_title('Rotated stiffness Q̄ vs ply angle — IM7/8552', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  Q̄₁₁ at 0°  = {Qb11[0]:.1f} GPa  (fibres carry load — very stiff)')
print(f'  Q̄₁₁ at 90° = {Qb11[-1]:.1f} GPa  (matrix carries load — much softer)')
print(f'  Q̄₆₆ peaks at 45° = {max(Qb66):.1f} GPa  (±45° plies provide shear stiffness)')
print(f'  Q̄₁₆ = 0 at 0° and 90°  (no coupling — use balanced ±θ pairs to keep it small)')

---
## 3. Classical Laminate Theory — The ABD Matrix

For a stack of N plies, CLT assembles the individual Q̄ contributions into the **ABD matrix**:

$$\begin{bmatrix} N \\ M \end{bmatrix} = \begin{bmatrix} A & B \\ B & D \end{bmatrix} \begin{bmatrix} \varepsilon_0 \\ \kappa \end{bmatrix}$$

| Matrix | Physical meaning | Formula |
|--------|-----------------|----------|
| **A** | Extensional stiffness | $A_{ij} = \sum_k \bar{Q}_{ij}^{(k)} t_k$ |
| **B** | Bending-extension coupling | $B_{ij} = \sum_k \bar{Q}_{ij}^{(k)} z_{mid,k} t_k$ |
| **D** | Bending stiffness | $D_{ij} = \sum_k \bar{Q}_{ij}^{(k)} (z_{1k}^3 - z_{0k}^3)/3$ |

For **symmetric laminates** (mirror image about mid-plane), **B = 0** exactly — membrane loads
don't cause bending, and moments don't cause in-plane strains. This is always enforced in our designs.

Reference: Jones (1999) §4.3

In [ ]:
from composite_panel import Ply, Laminate

# Build a quasi-isotropic [±45/0/90]s laminate — 8 plies at 0.125mm each
PLY_T  = 0.125e-3   # 0.125 mm per ply
angles = [-45, 0, 45, 90, 90, 45, 0, -45]   # symmetric stack
plies  = [Ply(mat, PLY_T, a) for a in angles]
lam    = Laminate(plies)

print(lam.summary())
print()

# Check B = 0 for symmetric laminate
B_max = np.max(np.abs(lam.B))
print(f'Max |B| = {B_max:.2e} N  (should be ~0 for symmetric laminate)')
print(f'Total thickness h = {lam.thickness*1e3:.3f} mm')
print(f'Effective Ex = {lam.Ex/1e9:.1f} GPa')
print(f'Effective Ey = {lam.Ey/1e9:.1f} GPa')
print()
print('For a quasi-isotropic laminate, Ex ≈ Ey (same stiffness in all directions)')
print(f'  Ratio Ey/Ex = {lam.Ey/lam.Ex:.4f}  (1.000 = perfectly isotropic)')

In [ ]:
# Compare A matrices for different layup families
# This shows how the choice of angles changes extensional stiffness

layups = {
    '[0]₈':           [0,0,0,0,0,0,0,0],
    '[±45/0/90]ₛ':    [-45,0,45,90,90,45,0,-45],    # quasi-isotropic
    '[±45]₂ₛ':        [-45,45,-45,45,45,-45,45,-45],  # shear-dominated
    '[90]₈':          [90,90,90,90,90,90,90,90],
}

print(f'  {"Layup":<20}  {"Ex [GPa]":>10}  {"Ey [GPa]":>10}  {"A66/A11":>10}')
print('  ' + '-'*55)
for name, ang in layups.items():
    l = Laminate([Ply(mat, PLY_T, a) for a in ang])
    A66_A11 = l.A[2,2] / l.A[0,0]
    print(f'  {name:<20}  {l.Ex/1e9:>10.1f}  {l.Ey/1e9:>10.1f}  {A66_A11:>10.3f}')
print()
print('Notes:')
print('  [0]₈        — all fibres aligned: max Ex, minimum Ey (very anisotropic)')
print('  [±45/0/90]ₛ — quasi-isotropic: Ex ≈ Ey (use when load direction unknown)')
print('  [±45]₂ₛ     — shear optimised: low Ex, Ey but high A66 shear stiffness')

---
## 4. CLT Response — Strains and Ply Stresses

Given applied loads **{N}** and moments **{M}**, CLT gives us the full stress state.

**Step 1 — Midplane response** (B = 0 for symmetric laminates):
$$\varepsilon_0 = [A]^{-1}\{N\}, \qquad \kappa = [D]^{-1}\{M\}$$

**Step 2 — Strain at ply mid-plane** (linear through thickness):
$$\varepsilon_{xy}(z) = \varepsilon_0 + z \cdot \kappa$$

**Step 3 — Stress in global frame** (using Q̄ for each ply):
$$\sigma_{xy} = [\bar{Q}_k] \, \varepsilon_{xy}(z_{mid,k})$$

**Step 4 — Stress in ply material frame** (rotate back using T(θ)):
$$\sigma_{12} = [T(\theta_k)] \, \sigma_{xy}$$

The ply-level stresses σ₁, σ₂, τ₁₂ are what the failure criterion uses.

Reference: Jones (1999) §4.4

In [ ]:
from composite_panel import supersonic_panel_loads

# Get a representative aerodynamic load state
# Mach 1.6 cruise at 15 km, alpha=3°, 80cm chord panel, 2.5g
loads = supersonic_panel_loads(
    mach=1.6, altitude_m=15000, alpha_deg=3.0,
    panel_chord=0.80, panel_span=0.50, n_load=2.5,
)
print('Applied running loads:')
print(f'  Nxx = {loads.Nxx/1e3:+.2f} kN/m  (spanwise compression)')
print(f'  Nyy = {loads.Nyy/1e3:+.2f} kN/m  (chordwise compression from aero pressure)')
print(f'  Nxy = {loads.Nxy/1e3:+.2f} kN/m  (in-plane shear from torsion)')
print(f'  Mxx = {loads.Mxx:+.3f} N·m/m  (panel bending from pressure)')

# Solve CLT
res = lam.response(N=loads.N, M=loads.M)

print()
print('CLT response:')
print(f'  ε₀ (midplane strains) = {[f"{e*1e6:.1f}" for e in res["eps0"]]} με')
print(f'  κ  (curvatures)       = {[f"{k:.4f}" for k in res["kappa"]]} 1/m')
print()
print('Ply-level stresses in material (1-2) frame:')
print(f'  {"Ply":>4}  {"θ [°]":>6}  {"σ₁ [MPa]":>10}  {"σ₂ [MPa]":>10}  {"τ₁₂ [MPa]":>10}')
print('  ' + '-'*50)
for k, ply in enumerate(lam.plies):
    sig = res['ply_stress_12'][k]
    print(f'  {k:>4}  {ply.angle:>6.0f}  {sig[0]/1e6:>10.1f}  {sig[1]/1e6:>10.1f}  {sig[2]/1e6:>10.1f}')

In [ ]:
# Visualise the through-thickness stress distribution
# σ₁ (fibre stress) varies by ply because each Q̄ is different

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
titles = ['σ₁  (fibre direction)', 'σ₂  (transverse)', 'τ₁₂  (shear)']
allowables_pos = [mat.F1t/1e6, mat.F2t/1e6, mat.F12/1e6]
allowables_neg = [-mat.F1c/1e6, -mat.F2c/1e6, -mat.F12/1e6]

for col, (ax, title, f_pos, f_neg) in enumerate(zip(axes, titles, allowables_pos, allowables_neg)):
    stresses = [res['ply_stress_12'][k][col] / 1e6 for k in range(len(lam.plies))]
    colors = [RED if s < 0 else BLUE for s in stresses]
    ax.barh(range(len(lam.plies)), stresses, color=colors, edgecolor=DIM)
    ax.axvline(0,     color=WHITE, lw=0.5, alpha=0.4)
    ax.axvline(f_pos, color=BLUE,  lw=1.2, linestyle='--', alpha=0.7, label=f'+allow. {f_pos:.0f}')
    ax.axvline(f_neg, color=RED,   lw=1.2, linestyle='--', alpha=0.7, label=f'-allow. {f_neg:.0f}')
    ax.set_yticks(range(len(lam.plies)))
    ax.set_yticklabels([f'{a}°' for a in angles], fontsize=8)
    ax.set_xlabel('[MPa]')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, framealpha=0.2)

plt.suptitle('[±45/0/90]ₛ  |  M1.6 @ 15km  |  Ply stresses in material frame',
             color=WHITE, fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print('Note: stresses vary dramatically between plies of the same angle at different')
print('z-positions (e.g. the two 90° plies) — this is because the moment Mxx adds')
print('a bending strain that is positive above and negative below the mid-plane.')

---
## 5. Tsai-Wu Failure Criterion

The **Tsai-Wu criterion** (1971) is a quadratic interaction formula that combines all
three stress components into a single failure index:

$$F_1\sigma_1 + F_2\sigma_2 + F_{11}\sigma_1^2 + F_{22}\sigma_2^2 + F_{66}\tau_{12}^2 + 2F_{12}\sigma_1\sigma_2 = 1 \quad \text{at failure}$$

The **Reserve Factor (RF)** answers: "by what factor can I scale up the current
load before this ply fails?"

Substituting RF·σ into the criterion gives a quadratic:
$$a \cdot RF^2 + b \cdot RF - 1 = 0 \quad \Rightarrow \quad RF = \frac{-b + \sqrt{b^2 + 4a}}{2a}$$

This closed-form **branch-free** formula is what makes the optimizer work — no if/else on
stress sign, so CasADi can differentiate through it.

Reference: Tsai & Wu (1971) J. Composite Materials 5(1), 58–80

In [ ]:
from composite_panel import check_laminate

# Evaluate Tsai-Wu RF for each ply
results = check_laminate(res, plies, criterion='tsai_wu', verbose=True)

In [ ]:
# Visualise the failure envelope in σ₁-σ₂ space for a 0° ply
# and plot where our actual stress state lands

# Tsai-Wu coefficients
F1  = 1/mat.F1t - 1/mat.F1c
F2  = 1/mat.F2t - 1/mat.F2c
F11 = 1/(mat.F1t * mat.F1c)
F22 = 1/(mat.F2t * mat.F2c)
F12i = -0.5 / np.sqrt(mat.F1t * mat.F1c * mat.F2t * mat.F2c)

# Build failure envelope (τ₁₂ = 0 for clarity)
s1_range = np.linspace(-mat.F1c/1e6, mat.F1t/1e6, 500)
# At each σ₁, solve for σ₂ on the failure envelope (quadratic in σ₂)
envelope_pos, envelope_neg = [], []
for s1 in s1_range:
    # F22·σ₂² + (F2 + 2F12i·s1)·σ₂ + (F11·s1² + F1·s1 - 1) = 0
    a_ = F22
    b_ = F2 + 2*F12i*s1
    c_ = F11*s1**2 + F1*s1 - 1
    disc = b_**2 - 4*a_*c_
    if disc >= 0:
        envelope_pos.append((-b_ + np.sqrt(disc)) / (2*a_) / 1e6)
        envelope_neg.append((-b_ - np.sqrt(disc)) / (2*a_) / 1e6)
    else:
        envelope_pos.append(np.nan)
        envelope_neg.append(np.nan)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(s1_range, envelope_pos, color=BLUE, lw=2, label='Tsai-Wu failure envelope')
ax.plot(s1_range, envelope_neg, color=BLUE, lw=2)
ax.fill_between(s1_range, envelope_neg, envelope_pos, alpha=0.1, color=BLUE, label='Safe region')

# Plot actual ply stress states
for k, r in enumerate(results):
    sig = res['ply_stress_12'][k]
    color = RED if r.rf < 1.0 else (GOLD if r.rf < 1.5 else TEAL)
    ax.scatter(sig[0]/1e6, sig[1]/1e6, color=color, s=80, zorder=5)
    ax.annotate(f'{angles[k]}° (RF={r.rf:.2f})', (sig[0]/1e6, sig[1]/1e6),
                textcoords='offset points', xytext=(8, 4), fontsize=7, color=color)

ax.axhline(0, color=DIM, lw=0.5)
ax.axvline(0, color=DIM, lw=0.5)
ax.set_xlabel('σ₁  [MPa]  (fibre direction)')
ax.set_ylabel('σ₂  [MPa]  (transverse)')
ax.set_title('Tsai-Wu failure envelope — IM7/8552  (τ₁₂ = 0 plane)', fontsize=11)
ax.legend(fontsize=9)

# Add colour legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=TEAL, markersize=9, label='RF ≥ 1.5  (OK)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=GOLD, markersize=9, label='1.0 ≤ RF < 1.5  (marginal)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=RED,  markersize=9, label='RF < 1.0  (FAIL)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper right')
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 6. Aerodynamic Loads — From Flight Condition to Running Loads

The loads driving structural sizing come from the flight condition.
For **supersonic to hypersonic** speeds, `panel_pressure()` selects the right model:

| Regime | Model | Formula |
|--------|-------|---------|
| Subsonic M < 0.85 | Prandtl-Glauert | ΔCp = 4α / √(1−M²) |
| Supersonic M 1.15–5 | Oblique shock | Exact Rankine-Hugoniot + expansion |
| Hypersonic M > 5 | Oblique shock | Same — naturally extends |
| Transonic 0.85–1.15 | Linear blend | Avoids M=1 singularity |

The pressure Δp is then converted to structural running loads [N/m]:
- **Nyy** = −Δp × chord/2  (chordwise compression, upper skin)
- **Nxx** = −M_bend / (box height × chord) × cos²(sweep)  (from elliptic bending moment)
- **Nxy** = 0.15|Nxx| + 0.10|Nyy|  (torsion + sweep — empirical)
- **Mxx** = Δp × chord² / 8  (simply-supported panel under uniform pressure)

In [ ]:
from composite_panel.aero_loads import panel_pressure, _isa

# Show how ΔCp and Δp vary with Mach at fixed altitude and alpha
alpha_deg = 3.0
alt_m     = 20_000   # 20 km
rho, a_sound = _isa(alt_m)

machs = np.concatenate([np.linspace(0.3, 0.84, 40), np.linspace(1.16, 10, 60)])
dCp_vals, dp_vals = [], []

for M in machs:
    q = 0.5 * rho * (M * a_sound)**2
    dp = panel_pressure(M, alpha_deg, q)
    dCp_vals.append(dp / q)
    dp_vals.append(dp / 1e3)   # kPa

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, vals, ylabel, title in zip(axes,
    [dCp_vals, dp_vals],
    ['ΔCp  [−]', 'Δp  [kPa]'],
    ['Pressure coefficient vs Mach', 'Net pressure vs Mach  (20 km, α=3°)']):
    ax.plot(machs[machs < 1], [v for M,v in zip(machs,vals) if M<1], color=BLUE, lw=2, label='Subsonic (PG)')
    ax.plot(machs[machs > 1], [v for M,v in zip(machs,vals) if M>1], color=RED,  lw=2, label='Supersonic/hypersonic')
    ax.axvspan(0.85, 1.15, alpha=0.15, color=GOLD, label='Transonic blend')
    ax.axvline(1.0, color=WHITE, lw=0.8, linestyle=':')
    ax.set_xlabel('Mach number')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

print('Key physics:')
print('  Subsonic: ΔCp rises toward M=1 (compressibility amplifies lift per degree AoA)')
print('  Supersonic: ΔCp falls as M increases (shock angle decreases, weaker pressure jump)')
print('  Δp peaks near M=1 despite ΔCp trend — because dynamic pressure q grows with M²')

---
## 7. The Optimizer — Minimum-Mass Laminate Design

Now we put it all together. `optimize_laminate()` formulates this problem:

$$\min_{t_k} \quad \rho \cdot 2 \sum_k t_k$$

$$\text{subject to:} \quad RF_k(t) \geq RF_{min} \quad \forall k$$
$$\quad\quad\quad\quad\quad t_k \geq t_{min}$$
$$\quad\quad\quad\quad\quad t_i = t_j \quad \text{(balance pairs)}$$

The optimizer is **IPOPT**, an industrial interior-point NLP solver, driven through
**CasADi's automatic differentiation** — exact Jacobians and Hessians, not finite differences.

**Why does this work?** Every step of the CLT + Tsai-Wu calculation is expressed using
`aerosandbox.numpy` — a CasADi-backed numpy drop-in. When `t_k` are optimization variables
(CasADi MX symbols), all downstream operations build a **symbolic computation graph**.
IPOPT differentiates through that graph automatically.

In [ ]:
from composite_panel.optimizer import optimize_laminate, detect_balance_pairs

# Define the half-stack (bottom half of symmetric laminate)
# Full laminate will be [0/+45/-45/90]s = 8 plies
angles_half = [0.0, 45.0, -45.0, 90.0]
pairs = detect_balance_pairs(angles_half)
print(f'Balance pairs detected: {pairs}  (ply {pairs[0][0]} = +45° paired with ply {pairs[0][1]} = -45°)')
print()

# Run the optimizer at Mach 1.6, 15km
loads_16 = supersonic_panel_loads(
    mach=1.6, altitude_m=15000, alpha_deg=3.0,
    panel_chord=0.80, panel_span=0.50, n_load=2.5,
)

result = optimize_laminate(
    N_loads         = loads_16.N,
    M_loads         = loads_16.M,
    mat             = mat,
    angles_half_deg = angles_half,
    rf_min          = 1.5,         # require 50% margin on every ply
    t_min           = 0.05e-3,     # 0.05 mm minimum gage
    t_init          = 0.125e-3,    # start from one standard ply
    balance_pairs   = pairs,
    rho_kg_m3       = 1600.0,
    verbose         = False,
)
print(result.summary())

In [ ]:
# Compare the un-optimized fixed layup vs the optimized layup
# Fixed: all plies at 0.125 mm  |  Optimized: IPOPT-found thicknesses

# Fixed-thickness laminate (QI, 0.125mm per ply)
fixed_lam = Laminate([Ply(mat, 0.125e-3, a) for a in [-45,0,45,90,90,45,0,-45]])
fixed_res = fixed_lam.response(N=loads_16.N, M=loads_16.M)
fixed_fails = check_laminate(fixed_res, fixed_lam.plies, criterion='tsai_wu', verbose=False)
fixed_min_rf = min(r.rf for r in fixed_fails)

print('Comparison — same layup, different thicknesses:')
print(f'  Fixed 0.125mm/ply:  h = {fixed_lam.thickness*1e3:.2f} mm,  '
      f'ρh = {1600*fixed_lam.thickness:.3f} kg/m²,  min RF = {fixed_min_rf:.3f}')
print(f'  Optimized:          h = {result.total_h*1e3:.2f} mm,  '
      f'ρh = {result.areal_density:.3f} kg/m²,  min RF = {result.min_tsai_wu_rf:.3f}')
print()
mass_saving = (1 - result.areal_density / (1600 * fixed_lam.thickness)) * 100
print(f'  Mass saving from optimization: {mass_saving:.1f}%')
print()
print('Optimized ply thicknesses (half-stack):')
for angle, t in zip(angles_half, result.t_half):
    print(f'  {angle:+6.1f}°  →  {t*1e3:.3f} mm')

In [ ]:
# Effect of the required RF on laminate mass
# As we demand a higher safety margin, we need more material

rf_values = np.linspace(1.0, 2.5, 20)
masses = []

for rf in rf_values:
    r = optimize_laminate(
        N_loads=loads_16.N, M_loads=loads_16.M, mat=mat,
        angles_half_deg=angles_half, balance_pairs=pairs,
        rf_min=float(rf), t_min=0.05e-3, t_init=0.125e-3,
        rho_kg_m3=1600.0, verbose=False,
    )
    masses.append(r.areal_density)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rf_values, masses, color=BLUE, lw=2.5, marker='o', markersize=5)
ax.axvline(1.5, color=GOLD, lw=1.5, linestyle=':', label='RF = 1.5  (typical design target)')
ax.fill_between(rf_values, min(masses), masses, alpha=0.12, color=BLUE)
ax.set_xlabel('Required reserve factor RF_min  [−]')
ax.set_ylabel('Areal mass  [kg/m²]')
ax.set_title('Mass vs structural margin — Pareto frontier  (M1.6 @ 15km)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

print('This is the Pareto frontier — the minimum mass achievable for each RF level.')
print('Below this curve: infeasible (impossible to meet RF at lower mass).')
print('Above this curve: suboptimal (more mass than needed).')

---
## 8. Adding Thermal Loads

At hypersonic speeds, aerodynamic heating causes the skin to reach temperatures
far above the cure temperature (~177°C for IM7/8552). The thermal mismatch between
fibre and matrix generates **thermal running loads** that add to the mechanical ones:

$$N_T = \sum_k \bar{Q}_k \, \bar{\alpha}_k \, \Delta T_k \, t_k$$

where $\bar{\alpha}_k$ is the transformed CTE vector and $\Delta T = T_{wall} - T_{cure}$.

Reference: Jones (1999) §5.4; Kassapoglou (2013) §8.2

In [ ]:
from composite_panel import IM7_8552_thermal, thermal_state_from_flight, aero_wall_temperature

pt = IM7_8552_thermal()
print('IM7/8552 CTE (coefficient of thermal expansion):')
print(f'  α₁ (fibre direction)  = {pt.alpha_1*1e6:.2f} µm/m/K  — near zero (fibres barely expand)')
print(f'  α₂ (transverse)       = {pt.alpha_2*1e6:.2f} µm/m/K  — ~95x larger! (matrix expansion)')
print(f'  T_cure                = {pt.T_cure - 273.15:.0f}°C  (stress-free reference temperature)')
print()

# Compare T_aw across the Mach range
machs_th = [0.8, 1.7, 2.4, 4.0, 5.0, 8.0]
print(f'  {"Mach":>5}  {"T_aw [°C]":>10}  {"ΔT vs cure [°C]":>18}  {"Thermal dominant?":>18}')
print('  ' + '-'*60)
for M in machs_th:
    T_aw_K = aero_wall_temperature(M, 20_000)
    dT = T_aw_K - pt.T_cure
    dominant = 'YES — thermal governs' if abs(dT) > 200 else 'no'
    print(f'  {M:>5.1f}  {T_aw_K-273.15:>10.0f}  {dT:>18.0f}  {dominant:>18}')

In [ ]:
# Compare mechanical-only vs thermal+mechanical optimization at M=5
from composite_panel.aero_loads import wing_panel_loads, WingGeometry

wing = WingGeometry(semi_span=4.5, root_chord=4.0, taper_ratio=0.25,
                    sweep_le_deg=50.0, t_over_c=0.04, mtow_n=120_000.0)

# Root panel loads at Mach 5
pl_m5 = wing_panel_loads(wing, eta=0.1, mach=5.0, altitude_m=20_000,
                          alpha_deg=3.0, n_load=2.5)

# Thermal state at Mach 5
ts_m5 = thermal_state_from_flight(5.0, 20_000, x_station=0.4 * wing.chord(0.1))

# Run without thermal
r_mech = optimize_laminate(
    N_loads=pl_m5.N, M_loads=pl_m5.M, mat=mat,
    angles_half_deg=angles_half, balance_pairs=pairs,
    rf_min=1.5, t_min=0.05e-3, t_init=0.15e-3,
    rho_kg_m3=1600.0, verbose=False,
)

# Run with thermal
r_therm = optimize_laminate(
    N_loads=pl_m5.N, M_loads=pl_m5.M, mat=mat,
    angles_half_deg=angles_half, balance_pairs=pairs,
    rf_min=1.5, t_min=0.05e-3, t_init=0.15e-3,
    thermal_state=ts_m5, ply_thermal=pt,
    rho_kg_m3=1600.0, verbose=False,
)

print(f'M5 root panel — mechanical only:      h = {r_mech.total_h*1e3:.2f} mm,  ρh = {r_mech.areal_density:.3f} kg/m²')
print(f'M5 root panel — thermal + mechanical: h = {r_therm.total_h*1e3:.2f} mm,  ρh = {r_therm.areal_density:.3f} kg/m²')
print(f'Thermal penalty: +{(r_therm.areal_density/r_mech.areal_density - 1)*100:.1f}%')

---
## 9. Wing-Level Sizing — Spanwise Variation

`optimize_wing()` calls `optimize_laminate()` independently at N spanwise stations η = y/b.
The load varies from root to tip — giving the characteristic **spanwise taper** of a composite wing skin.

The mass is then integrated along the semi-span:
$$m_{skin} = \int_0^b \rho \cdot h(\eta) \cdot c(\eta) \, dy \approx \sum_i \rho h_i \cdot c_i \cdot \Delta y_i$$

In [ ]:
from composite_panel.optimizer import optimize_wing

# Size the wing at two Mach numbers and compare
common_kwargs = dict(
    wing=wing, altitude_m=20_000, alpha_deg=3.0, mat=mat,
    angles_half_deg=angles_half, n_load=2.5, n_stations=10,
    rf_min=1.5, t_min=0.05e-3, t_init=0.15e-3,
    balance_pairs=pairs, rho_kg_m3=1600.0,
)

print('Sizing at Mach 1.7...')
wr_17 = optimize_wing(mach=1.7, **common_kwargs)

print('Sizing at Mach 5.0...')
wr_50 = optimize_wing(mach=5.0, **common_kwargs)

In [ ]:
# Plot spanwise thickness taper and load distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

etas = wr_17.etas

# Thickness
ax = axes[0]
ax.plot(etas, wr_17.thicknesses * 1e3, color=BLUE, lw=2.5, marker='o', ms=5, label='M1.7')
ax.plot(etas, wr_50.thicknesses * 1e3, color=RED,  lw=2.5, marker='o', ms=5, label='M5.0')
ax.set_xlabel('η = y/b  [−]')
ax.set_ylabel('Laminate thickness  [mm]')
ax.set_title('Spanwise thickness taper', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True)

# Nxx load (bending — same at both Mach since it comes from MTOW)
ax = axes[1]
ax.plot(etas, wr_17.Nxx / 1e3, color=BLUE, lw=2, label='Nxx M1.7')
ax.plot(etas, wr_50.Nxx / 1e3, color=RED,  lw=2, label='Nxx M5.0')
ax.plot(etas, wr_17.Nyy / 1e3, color=BLUE, lw=2, linestyle='--', label='Nyy M1.7')
ax.plot(etas, wr_50.Nyy / 1e3, color=RED,  lw=2, linestyle='--', label='Nyy M5.0')
ax.axhline(0, color=DIM, lw=0.5)
ax.set_xlabel('η = y/b  [−]')
ax.set_ylabel('Running load  [kN/m]')
ax.set_title('Spanwise load distribution\n(solid=Nxx bending, dashed=Nyy pressure)', fontsize=9)
ax.legend(fontsize=8)
ax.grid(True)

# Mass penalty
ax = axes[2]
pct = (wr_50.areal_densities - wr_17.areal_densities) / wr_17.areal_densities * 100
ax.plot(etas, pct, color=GOLD, lw=2.5, marker='o', ms=5)
ax.axhline(0, color=DIM, lw=0.5)
ax.fill_between(etas, 0, pct, alpha=0.15, color=GOLD)
ax.set_xlabel('η = y/b  [−]')
ax.set_ylabel('Mass penalty  [%]')
ax.set_title('M5.0 vs M1.7 areal density penalty', fontsize=10)
ax.grid(True)

plt.suptitle(f'Wing sizing comparison  |  M1.7 ({wr_17.total_skin_mass:.1f} kg) vs'
             f' M5.0 ({wr_50.total_skin_mass:.1f} kg)',
             color=WHITE, fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Semi-span upper-skin mass:')
print(f'  M1.7: {wr_17.total_skin_mass:.1f} kg')
print(f'  M5.0: {wr_50.total_skin_mass:.1f} kg  (+{(wr_50.total_skin_mass/wr_17.total_skin_mass-1)*100:.0f}%)')

---
## 10. Loads Database — Multi-Source Load Management

`LoadsDatabase` stores, filters, and envelopes load cases from any source (aero analysis, test, hand-calc). It reads/writes CSV so it integrates with existing data pipelines.

This is the bridge between upstream aero tools and the structural optimizer — instead of hardcoding a single `[Nxx, Nyy, Nxy]` vector, you accumulate all flight conditions and let the optimizer see them all at once.

In [ ]:
from composite_panel import LoadCase, LoadsDatabase
import numpy as np
import tempfile, os

# Build a small database manually (in practice: loaded from flight_envelope.csv)
cases = [
    LoadCase("M08_2p5g_root",  Nxx=-380e3, Nyy=-18e3, Nxy=60e3,  source="aero", eta=0.15),
    LoadCase("M17_2p5g_root",  Nxx=-490e3, Nyy=-32e3, Nxy=80e3,  source="aero", eta=0.15),
    LoadCase("M08_gust_root",  Nxx=-320e3, Nyy=-15e3, Nxy=50e3,  source="aero", eta=0.15),
    LoadCase("M08_2p5g_mid",   Nxx=-210e3, Nyy=-14e3, Nxy=35e3,  source="aero", eta=0.45),
    LoadCase("M17_2p5g_mid",   Nxx=-270e3, Nyy=-24e3, Nxy=45e3,  source="aero", eta=0.45),
    LoadCase("M08_2p5g_tip",   Nxx=-80e3,  Nyy=-10e3, Nxy=15e3,  source="aero", eta=0.75),
]
db = LoadsDatabase(cases)
db.print_summary()

# Filter to root station and get component-wise envelope
root_db = db.filter_eta(0.15)
env = root_db.envelope_case()
print(f"\nEnvelope at η=0.15: Nxx={env.Nxx/1e3:.0f} kN/m  Nyy={env.Nyy/1e3:.0f} kN/m  Nxy={env.Nxy/1e3:.0f} kN/m")

# CSV round-trip
csv_path = os.path.join(tempfile.gettempdir(), 'demo_loads.csv')
db.to_csv(csv_path)
db2 = LoadsDatabase.from_csv(csv_path)
print(f"Round-trip: {len(db2.cases)} cases reloaded from {csv_path}")


---
## 11. Trim Analysis — Equilibrium Angle of Attack

In an MDO loop, angle of attack is not a guess — it is solved from the lift balance:

$$C_{L,\text{req}} = \frac{n \cdot W}{q_\infty S_{\text{ref}}}, \qquad \alpha_{\text{trim}} = \frac{C_{L,\text{req}}}{C_{L\alpha}}$$

`lift_curve_slope()` dispatches by Mach regime:
- **Subsonic** (M ≤ 0.85): Prandtl-Glauert 2D + Helmbold finite-wing correction
- **Supersonic** (M ≥ 1.15): Ackeret 2D + Jones (1947) supersonic finite-wing
- **Transonic** (0.85–1.15): linear blend to avoid the M=1 singularity


In [ ]:
from composite_panel import WingGeometry, trim_alpha, trim_table, lift_curve_slope
import numpy as np

wing = WingGeometry(
    semi_span=7.5, root_chord=3.5, taper_ratio=0.3,
    sweep_le_deg=35.0, t_over_c=0.05, mtow_n=120e3,
)

machs = np.linspace(0.3, 2.5, 200)
cla_2d = [lift_curve_slope(M) for M in machs]
cla_3d = [lift_curve_slope(M, aspect_ratio=6.0, taper_ratio=0.3, sweep_le_deg=35.0)
          for M in machs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(machs, cla_2d, 'b-', label='2D (infinite wing)')
axes[0].plot(machs, cla_3d, 'r-', label='3D (AR=6, Λ=35°)')
axes[0].axvspan(0.85, 1.15, alpha=0.15, color='gray', label='transonic blend')
axes[0].set_xlabel('Mach'); axes[0].set_ylabel('CLα [rad⁻¹]')
axes[0].set_title('Lift Curve Slope vs Mach'); axes[0].legend(); axes[0].grid(True)

for n in [1.0, 2.5, 3.5]:
    states = trim_table(wing, np.linspace(0.5, 2.5, 30), altitude_m=10000, n_load=n)
    axes[1].plot([s.mach for s in states], [s.alpha_deg for s in states], label=f'n={n}g')
axes[1].set_xlabel('Mach'); axes[1].set_ylabel('α trim [deg]')
axes[1].set_title('Trim AoA vs Mach (alt=10 km)'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

ts = trim_alpha(wing, mach=0.8, altitude_m=8000, n_load=2.5)
print(ts)


---
## 12. Static Aeroelastic Analysis — Flexible Wing Response

A real wing bends under load. For a swept wing, upward bending causes a nose-down geometric twist at the tip — **aeroelastic washout** — which reduces the effective angle of attack and relieves tip loads.

The iterative solution sequence:
```
α_rigid → loads(y) → q(y) → θ(y) = ∫M/EI dy  →  Δα(y) = −θ·tan(Λ)  →  α_eff(y)  →  repeat
```

`wing_bending_stiffness()` derives EI(y) from the laminate CLT A-matrix using a thin-walled box model:
```
E_eff = A₁₁ / h_skin
EI(y) = 2 × E_eff × b_box(y) × (h_box(y)/2)²     ∝  A₁₁ × chord³ × (t/c)²
```

This directly connects the structural optimizer output (A₁₁) to the aeroelastic response, closing the aero-structures MDO loop.

In [ ]:
from composite_panel import (IM7_8552, Ply, Laminate,
                              static_aeroelastic, wing_bending_stiffness)
import numpy as np

mat = IM7_8552()
# wing defined in cell 11 above

def run_aero(angles, label):
    plies = [Ply(mat, 0.125e-3, th) for th in angles]
    lam   = Laminate(plies)
    res   = static_aeroelastic(
        wing, mach=0.8, altitude_m=8000,
        alpha_rigid_deg=5.25, n_load=2.5,
        laminate_A11=lam.A[0, 0], laminate_h=lam.thickness,
    )
    return res, label

configs = [
    run_aero([0] * 16,                              "[0]₁₆  (stiff)"),
    run_aero([-45, 0, 45, 90] * 2 + [90, 45, 0, -45] * 2, "QI 16-ply  (flexible)"),
    run_aero([0, 90] * 4 + [90, 0] * 4,            "[0/90]₄s  (moderate)"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for res, label in configs:
    axes[0].plot(res.etas, res.alpha_eff - res.alpha_rigid, label=label)
    axes[1].plot(res.etas, [L.Nyy / 1e3 for L in res.loads_elastic], '--')
    axes[1].plot(res.etas, [L.Nyy / 1e3 for L in res.loads_rigid],   ':')
    axes[2].plot(res.etas, res.bending_slope * 1e3, label=label)

axes[0].set_xlabel('η'); axes[0].set_ylabel('Δα [deg]')
axes[0].set_title('Aeroelastic Twist (washout)'); axes[0].legend(fontsize=7); axes[0].grid(True)
axes[1].set_xlabel('η'); axes[1].set_ylabel('Nyy [kN/m]')
axes[1].set_title('Rigid (···) vs Elastic (- -) Nyy'); axes[1].grid(True)
axes[2].set_xlabel('η'); axes[2].set_ylabel('θ [mrad]')
axes[2].set_title('Bending Slope'); axes[2].legend(fontsize=7); axes[2].grid(True)
plt.tight_layout(); plt.show()

print(f"{"Layup":<28} {"tip deflection":>16}  {"aeroelastic relief":>18}")
print('-' * 66)
for res, label in configs:
    print(f"{label:<28} {res.tip_deflection*100:>14.1f} cm  {res.aeroelastic_relief*100:>+17.1f}%")


---
## Summary

Here's the full data flow in one diagram:

```
Flight condition (Mach, altitude, alpha, MTOW)
          │
          ▼  panel_pressure()  +  elliptic bending moment
    running loads  [Nxx, Nyy, Nxy, Mxx]  [N/m]
          │
          │  + N_T, M_T  from thermal_resultants()  (if hot)
          ▼
    optimize_laminate()
          │
          ├─ design variables:  t_k  (ply thicknesses)  [CasADi MX]
          ├─ Q̄_k(θ_k)          →  rotated ply stiffness
          ├─ A(t), D(t)         →  ABD matrices  [nonlinear in t via z³]
          ├─ ε₀ = A⁻¹N,  κ = D⁻¹M  →  midplane response
          ├─ σ₁₂,k              →  ply stresses
          ├─ RF_k ≥ 1.5         →  Tsai-Wu strength constraint
          ├─ RF_buckle ≥ 1.0    →  Whitney buckling constraint (optional)
          ├─ minimize ρ·2·Σt    →  mass objective
          └─ IPOPT (exact AD via CasADi)
                    │
                    ▼
            t*  →  OptimizationResult  (h, ρh, min_RF)
```

### What to explore next
- **Change `rf_min`** in Section 7 and see how the mass changes
- **Change the layup** (e.g. `[0, 0, 45, -45]`) and see which plies govern failure
- **Enable `optimize_angles=True`** in `optimize_laminate()` to let IPOPT find the best angles
- **Try Mach 8** in Section 9 — the thermal loads dominate
- **Add buckling** by passing `panel_a=0.5, panel_b=0.2, buckle_rf_min=1.0`